# Serve Qwen3-8B on a Colab A100 (remote inference for SWE-agent)

This notebook runs **only the neural-network inference**. It serves `Qwen/Qwen3-8B`
with vLLM behind an OpenAI-compatible API and exposes it through a public
Cloudflare tunnel.

**Next step (mini track, no Docker):** open
[`mini_agent_colab.ipynb`](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/mini_agent_colab.ipynb)
on a **CPU** runtime, paste the `MODEL_API_BASE` printed below, and run all cells.

**Runtime:** set **Runtime → Change runtime type → A100 GPU** before running.

> The same A100 is also where you run the GPU **projection** step
> (`stage2.extract.project_steps`) after ingesting trajectories — that step needs
> raw residual-stream activations and cannot go through this API.

In [1]:
# 0. Confirm we actually have a GPU (expect A100 40GB).
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# 1. Install a CUDA-12 build of vLLM.
import subprocess, sys

!pip -q uninstall -y vllm
!pip -q install "vllm==0.11.0"
# Colab preinstalls a newer transformers that removed all_special_tokens_extended,
# which vLLM 0.11.0's tokenizer-caching shim still calls. Force the compatible version.
!pip -q install --force-reinstall "transformers==4.55.4"
# vLLM's numba dependency (used for n-gram spec decoding) requires NumPy<=2.2.
!pip -q install "numpy<2.3"

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

check = subprocess.run(
    [sys.executable, "-c", "import vllm._C; import vllm; print('vLLM', vllm.__version__, 'C-ext OK')"],
    capture_output=True, text=True,
)
print(check.stdout or check.stderr)
assert check.returncode == 0, "vLLM C extension failed to load — see error above"
print(f'Will invoke vLLM via: {sys.executable} -m vllm.entrypoints.openai.api_server')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 155.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/7

In [3]:
# 2. Config. enable_thinking=False MUST match the local replay/projection step
#    (stage2 defaults use enable_thinking: false), or activations won't line up.
MODEL = "Qwen/Qwen3-8B"
SERVED_NAME = "Qwen3-8B"   # must equal the suffix in MODEL_NAME (hosted_vllm/Qwen3-8B)
PORT = 8000
MAX_MODEL_LEN = 32768
API_KEY = "EMPTY"          # set a real token here AND in MODEL_API_KEY locally to lock down the tunnel

In [4]:
# 3. Launch vLLM as a Python module (bypasses the stale system CLI binary).
#
# NOTE: enable_thinking is NOT a server-launch flag in this vLLM version
# (--chat-template-kwargs at the CLI level errors with "unrecognized arguments").
# It must be passed per-request as "chat_template_kwargs": {"enable_thinking": false}
# in the JSON body of each /v1/chat/completions call — see cell 6 and agent_loop.py.
import subprocess, sys, os, time

log = open("vllm.log", "w")
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),
]
if API_KEY and API_KEY != "EMPTY":
    cmd += ["--api-key", API_KEY]
print(" ".join(cmd))
vllm_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("vLLM starting (pid %d); first run downloads ~16GB of weights..." % vllm_proc.pid)

/usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B --served-model-name Qwen3-8B --dtype bfloat16 --max-model-len 32768 --port 8000
vLLM starting (pid 11979); first run downloads ~16GB of weights...


In [5]:
# 4. Wait until the server answers /v1/models (model load can take a few minutes).
import requests, time

headers = {} if API_KEY in ("", "EMPTY") else {"Authorization": f"Bearer {API_KEY}"}
url = f"http://localhost:{PORT}/v1/models"
for _ in range(120):  # up to ~10 min
    if vllm_proc.poll() is not None:
        raise RuntimeError("vLLM exited early — see tail below:\n" + open("vllm.log").read()[-3000:])
    try:
        r = requests.get(url, headers=headers, timeout=5)
        if r.ok:
            print("vLLM is up:", r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become ready; check vllm.log")

vLLM is up: {'object': 'list', 'data': [{'id': 'Qwen3-8B', 'object': 'model', 'created': 1782688250, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-8B', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-754fe85354c84176bccf482c89c7c163', 'object': 'model_permission', 'created': 1782688250, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [6]:
print(open('vllm.log').read()[-12000:])

1.06it/s]
(EngineCore_DP0 pid=12295) 
(EngineCore_DP0 pid=12295) INFO 06-28 23:09:39 [default_loader.py:267] Loading weights took 4.82 seconds
(EngineCore_DP0 pid=12295) INFO 06-28 23:09:40 [gpu_model_runner.py:2653] Model loading took 15.2683 GiB and 70.345354 seconds
(EngineCore_DP0 pid=12295) INFO 06-28 23:09:50 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/c30c9b32de/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=12295) INFO 06-28 23:09:50 [backends.py:559] Dynamo bytecode transform time: 9.11 s
(EngineCore_DP0 pid=12295) INFO 06-28 23:09:55 [backends.py:197] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=12295) INFO 06-28 23:10:24 [backends.py:218] Compiling a graph for dynamic shape takes 33.97 s
(EngineCore_DP0 pid=12295) INFO 06-28 23:10:33 [monitor.py:34] torch.compile takes 43.08 s in total
(EngineCore_DP0 pid=12295) INFO 06-28 23:10:35 [gpu_worker.py:298] Available KV cache memory: 54.61 GiB
(EngineCore_DP

In [9]:
# 5. Open a public tunnel to the local vLLM port and print the URL to use locally.
import subprocess, re, time

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
public = None
for _ in range(60):
    time.sleep(2)
    txt = open("cloudflared.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        public = m.group(0)
        break
if not public:
    raise RuntimeError("No tunnel URL yet; check cloudflared.log")

print("\n" + "=" * 70)
print("Remote model endpoint is LIVE.\n")
print("For the mini-agent track (Colab CPU, no Docker):")
print("  1. Open mini_agent_colab.ipynb in a new tab (CPU runtime)")
print("  2. Paste this into the config cell:\n")
print(f'     MODEL_API_BASE = "{public}/v1"')
print("\nKeep this tab open until the mini batch finishes.")
print("=" * 70)


Remote model endpoint is LIVE.

For the mini-agent track (Colab CPU, no Docker):
  1. Open mini_agent_colab.ipynb in a new tab (CPU runtime)
  2. Paste this into the config cell:

     MODEL_API_BASE = "https://component-sandwich-affecting-volumes.trycloudflare.com/v1"

Keep this tab open until the mini batch finishes.


In [10]:
# 6. Smoke-test the public endpoint end-to-end (same path SWE-agent will use).
# enable_thinking=False is passed per-request via chat_template_kwargs — it MUST
# match what agent_loop.py sends, or activations won't line up with the value axis.
import requests
r = requests.post(
    f"{public}/v1/chat/completions",
    headers=headers,
    json={
        "model": SERVED_NAME,
        "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
        "max_tokens": 8,
        "temperature": 0.0,
        "chat_template_kwargs": {"enable_thinking": False},
    },
    timeout=60,
)
print(r.status_code, r.json()["choices"][0]["message"]["content"])

200 OK


In [11]:
# 7. Keep this cell running to hold the tunnel + server open during the batch.
#    Interrupt it (or close the runtime) to shut everything down.
import time
try:
    while True:
        if vllm_proc.poll() is not None:
            print("vLLM exited; tail of log:\n", open("vllm.log").read()[-2000:])
            break
        time.sleep(30)
except KeyboardInterrupt:
    print("Shutting down vLLM + tunnel...")
    tunnel.terminate()
    vllm_proc.terminate()

Shutting down vLLM + tunnel...
